# 04 — Candidate Generation & Oracle Analysis

Generate N=32 candidates per question, execute & fingerprint, compute oracle metrics.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path


def bootstrap_repo():
    configured = os.environ.get('SLM_PRJ_PATH')
    candidates = [Path(configured).expanduser()] if configured else []
    current = Path.cwd().resolve()
    candidates.extend([current, *current.parents,
                       Path('/kaggle/working/SLM-PRJ'),
                       Path('/kaggle/working/SLMSQL_project')])
    for candidate in candidates:
        if (candidate / 'pyproject.toml').is_file() and (candidate / 'benchmarks').is_dir():
            return candidate
    target = Path('/kaggle/working/SLM-PRJ')
    url = os.environ.get('SLM_PRJ_REPO_URL', 'https://github.com/minhadcaz/SLMSQL_project.git')
    subprocess.run(['git', 'clone', url, str(target)], check=True)
    return target

REPO_DIR = bootstrap_repo()
sys.path.insert(0, str(REPO_DIR))
from scripts.kaggle_utils import prepare_repo_import, resolve_data_dir, load_kaggle_secrets, download_checkpoint
REPO_DIR = prepare_repo_import(REPO_DIR)
DATA_DIR = resolve_data_dir(REPO_DIR)

BIRD_DIR = DATA_DIR / 'bird'
print(f'Repo: {REPO_DIR}')

## Generate Candidates (GPU required)

In [ ]:
from benchmarks.bird_loader import load_bird_dev, get_schema
from research.candidate_generator import generate_candidates, save_candidates

model_path_setting = os.environ.get('MODEL_PATH', '').strip()
if model_path_setting:
    MODEL_PATH = Path(model_path_setting)
else:
    metadata_path = REPO_DIR / 'results' / 'lora' / 'best_checkpoint.json'
    if metadata_path.is_file():
        MODEL_PATH = Path(json.loads(metadata_path.read_text(encoding='utf-8'))['merged_path'])
    else:
        MODEL_PATH = REPO_DIR / 'results' / 'lora' / 'best_adapter' / 'merged'
OUTPUT = REPO_DIR / 'results' / 'candidates' / 'candidates_dev.jsonl'
limit = int(os.environ.get('BIRD_LIMIT', '0'))

all_bird_dev = load_bird_dev(BIRD_DIR)
bird_dev = all_bird_dev if limit == 0 else all_bird_dev[:limit]
if not MODEL_PATH.exists() and os.environ.get('HF_REPO_ID', '').strip():
    load_kaggle_secrets()
    MODEL_PATH = download_checkpoint(local_dir=REPO_DIR / 'results' / 'lora' / 'restored')
if not MODEL_PATH.exists():
    raise FileNotFoundError(f'Model not found: {MODEL_PATH}. Set MODEL_PATH or HF_REPO_ID to a merged checkpoint.')

prompts = []
for record in bird_dev:
    schema = get_schema(BIRD_DIR, record['db_id'])
    prompts.append({**record, 'schema': schema})

n_candidates = int(os.environ.get('N_CANDIDATES', '32'))
print(f'Generating {n_candidates} candidates for {len(prompts)} questions...')
results = generate_candidates(
    str(MODEL_PATH), prompts, n_candidates=n_candidates,
    batch_size=int(os.environ.get('GEN_BATCH_SIZE', '8')),
)
save_candidates(results, OUTPUT)
print(f'Saved to {OUTPUT}')


## Execute & Fingerprint

In [ ]:
from research.exec_runner import execute_candidates
from benchmarks.bird_loader import get_db_path

EXEC_OUTPUT = REPO_DIR / 'results' / 'candidates' / 'exec_dev.jsonl'
stats = execute_candidates(
    REPO_DIR / 'results' / 'candidates' / 'candidates_dev.jsonl',
    lambda db_id: get_db_path(BIRD_DIR, db_id),
    EXEC_OUTPUT,
    timeout=int(os.environ.get('SQL_TIMEOUT', '30')),
)
print(json.dumps(stats, indent=2))


## Oracle Analysis

In [ ]:
from research.oracle_analysis import load_exec_results, pass_at_k, sc_at_k, top2_coverage_at_k, analyze_by_group

records = load_exec_results(EXEC_OUTPUT)
db_path_fn = lambda db_id: get_db_path(BIRD_DIR, db_id)

print('=== Oracle Metrics ===')
for k in [4, 8, 16, 32]:
    p = pass_at_k(records, db_path_fn, k)
    s = sc_at_k(records, db_path_fn, k)
    t = top2_coverage_at_k(records, db_path_fn, k)
    print(f'k={k:2d}  pass@k={p:.4f}  SC@k={s:.4f}  top2@k={t:.4f}')

In [ ]:
results_by_diff = analyze_by_group(records, db_path_fn, [4, 8, 16, 32], 'difficulty')
for group, metrics in results_by_diff.items():
    print(f'\n{group} (n={metrics["count"]})')
    for k, v in sorted(metrics.items()):
        if k != 'count':
            print(f'  {k}: {v}')

## Calibration Analysis

In [ ]:
from research.calibration import compute_calibration_data, rank_features

calib_data = compute_calibration_data(records, db_path_fn)
print(f'Calibration rows: {len(calib_data)}')

print('\n=== Feature AUC Ranking ===')
for name, auc in rank_features(calib_data):
    print(f'  {name:20s}  AUC={auc:.4f}')
